# Limpieza — Fase 1: Identificadores y variables de dominio cerrado


In [1]:
import os
import sys
import re

import pandas as pd

sys.path.append(os.path.join("..", "scripts"))
from limpieza_utils import (
    CARPETA_DATA, CARPETA_INTERIM, es_vacio, normalizar_espacios,
    marcar_faltantes_como_na, RegistroTransformaciones,
)

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

RUTA_CRUDO = os.path.join(CARPETA_DATA, "establecimientos_diversificado_crudo.csv")
RUTA_CATALOGO = os.path.join(CARPETA_DATA, "catalogo_departamentos_municipios.csv")

df_raw = pd.read_csv(RUTA_CRUDO, dtype=str, keep_default_na=False, encoding="utf-8-sig")
catalogo = pd.read_csv(RUTA_CATALOGO, dtype=str, keep_default_na=False, encoding="utf-8-sig")

log = RegistroTransformaciones("Fase 1 - Identificadores")

print(f"Filas crudas: {len(df_raw)}  |  Columnas: {df_raw.shape[1]}")


Filas crudas: 11890  |  Columnas: 18


## Paso 0 — Descartar filas 100% vacías (artefacto de extracción)

In [2]:
cols_dato = [c for c in df_raw.columns if c != "ARCHIVO_ORIGEN"]
vacias = df_raw[cols_dato].apply(lambda col: col.map(es_vacio)).all(axis=1)

n_antes = len(df_raw)
df = df_raw.loc[~vacias].copy().reset_index(drop=True)
n_despues = len(df)

assert n_antes - n_despues == 23, f"Se esperaban exactamente 23 filas vacías, se encontraron {n_antes - n_despues}"

log.registrar(
    "(todas)", "23 filas 100% vacías (una por archivo/departamento, artefacto del GridView de la fuente)",
    "Eliminar filas completamente vacías", n_antes - n_despues,
    "No aportan ningún dato y distorsionan cualquier estadística; documentadas en el diagnóstico antes de eliminarlas.",
)
print(f"Filas: {n_antes} -> {n_despues}")


[Fase 1 - Identificadores] (todas): Eliminar filas completamente vacías -> 23 registros afectados
Filas: 11890 -> 11867


## CODIGO

Ya es 100% consistente en el crudo (patrón `##-##-####-##`, sin repetidos). Aquí solo se **verifica** con código — no se transforma ningún valor.


In [3]:
patron_codigo = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
cumple = df["CODIGO"].map(lambda x: bool(patron_codigo.match(x.strip())))

assert cumple.all(), f"{(~cumple).sum()} filas no cumplen el patrón de CODIGO"
assert df["CODIGO"].is_unique, "CODIGO tiene valores repetidos"
assert df["CODIGO"].dtype == object, "CODIGO debe mantenerse como texto (no numérico)"

log.registrar(
    "CODIGO", "Ninguno (ya cumple el patrón y es único en el 100% de las filas)",
    "Sin cambios; se verifica el patrón ##-##-####-## y la unicidad como prueba de calidad", 0,
    "No hay nada que corregir; se documenta la verificación para la Actividad 7.",
)
print("CODIGO: OK, 100% cumple el patrón y es único.")


[Fase 1 - Identificadores] CODIGO: Sin cambios; se verifica el patrón ##-##-####-## y la unicidad como prueba de calidad -> 0 registros afectados
CODIGO: OK, 100% cumple el patrón y es único.


## DISTRITO

Conviven varios formatos + 4.48% vacío. Se deja como texto categórico libre (no se fuerza un único patrón, ver riesgos en el plan) y se marca lo vacío como `NA` real.


In [4]:
antes_vacio = df["DISTRITO"].map(es_vacio).sum()

df["DISTRITO"] = normalizar_espacios(df["DISTRITO"])
df["DISTRITO"] = marcar_faltantes_como_na(df["DISTRITO"], tratar_placeholders=False)

despues_na = df["DISTRITO"].isna().sum()
assert despues_na == antes_vacio, f"Los NA de DISTRITO ({despues_na}) no coinciden con los vacíos originales ({antes_vacio})"
assert not df["DISTRITO"].dropna().str.contains(r"^\s|\s$", regex=True).any(), "Quedaron espacios al borde en DISTRITO"

log.registrar(
    "DISTRITO", f"{antes_vacio} celdas vacías; 3 formatos distintos conviviendo en el campo",
    "Trim/colapso de espacios y conversión de vacíos a NA explícito; se preserva como texto libre (no se fuerza un patrón único)",
    antes_vacio,
    "Forzar un solo formato numérico inventaría dígitos que no están en la fuente (ver docs/02_Plan_de_Limpieza.md).",
)
print(f"DISTRITO: {despues_na} valores marcados NA.")


[Fase 1 - Identificadores] DISTRITO: Trim/colapso de espacios y conversión de vacíos a NA explícito; se preserva como texto libre (no se fuerza un patrón único) -> 532 registros afectados
DISTRITO: 532 valores marcados NA.


## DEPARTAMENTO

Validado en el diagnóstico: 100% de los valores existe en el catálogo oficial del propio sitio (viene de un `<select>`, no es texto libre). Se normaliza defensivamente.


In [5]:
deptos_catalogo = set(catalogo["DEPARTAMENTO"].str.upper().str.strip())

df["DEPARTAMENTO"] = normalizar_espacios(df["DEPARTAMENTO"]).str.upper()

fuera_dominio = ~df["DEPARTAMENTO"].isin(deptos_catalogo)
assert fuera_dominio.sum() == 0, f"{fuera_dominio.sum()} filas con DEPARTAMENTO fuera del catálogo oficial"

log.registrar(
    "DEPARTAMENTO", "Ninguno (100% dentro del catálogo oficial del propio sitio)",
    "Trim + mayúsculas defensivo; se valida contra data/catalogo_departamentos_municipios.csv", 0,
    "Campo de <select> en la fuente, no texto libre digitado; se mantienen GUATEMALA y CIUDAD CAPITAL como categorías separadas (convención real de la fuente, ver docs/02).",
)
print("DEPARTAMENTO: OK, 100% dentro del catálogo.")


[Fase 1 - Identificadores] DEPARTAMENTO: Trim + mayúsculas defensivo; se valida contra data/catalogo_departamentos_municipios.csv -> 0 registros afectados
DEPARTAMENTO: OK, 100% dentro del catálogo.


## MUNICIPIO

Igual que DEPARTAMENTO: se valida el par `(DEPARTAMENTO, MUNICIPIO)` completo contra el catálogo.


In [6]:
pares_catalogo = set(zip(catalogo["DEPARTAMENTO"].str.upper().str.strip(), catalogo["MUNICIPIO"].str.upper().str.strip()))

df["MUNICIPIO"] = normalizar_espacios(df["MUNICIPIO"]).str.upper()

pares_datos = list(zip(df["DEPARTAMENTO"], df["MUNICIPIO"]))
fuera_dominio_mun = [p for p in pares_datos if p not in pares_catalogo]
assert len(fuera_dominio_mun) == 0, f"{len(fuera_dominio_mun)} filas con (DEPARTAMENTO, MUNICIPIO) fuera del catálogo"

log.registrar(
    "MUNICIPIO", "Ninguno (100% de los pares (DEPARTAMENTO, MUNICIPIO) existen en el catálogo oficial)",
    "Trim + mayúsculas defensivo; se valida el par completo contra el catálogo", 0,
    "Mismo argumento que DEPARTAMENTO: campo de <select>, no texto libre.",
)
print("MUNICIPIO: OK, 100% dentro del catálogo (validado junto con DEPARTAMENTO).")


[Fase 1 - Identificadores] MUNICIPIO: Trim + mayúsculas defensivo; se valida el par completo contra el catálogo -> 0 registros afectados
MUNICIPIO: OK, 100% dentro del catálogo (validado junto con DEPARTAMENTO).


## DEPARTAMENTAL

In [7]:
antes_unicos = df["DEPARTAMENTAL"].nunique()

df["DEPARTAMENTAL"] = normalizar_espacios(df["DEPARTAMENTAL"]).str.upper()
df["DEPARTAMENTAL"] = marcar_faltantes_como_na(df["DEPARTAMENTAL"], tratar_placeholders=False)

despues_unicos = df["DEPARTAMENTAL"].dropna().nunique()
assert despues_unicos <= antes_unicos, "La normalización no debería aumentar el número de categorías"
assert not df["DEPARTAMENTAL"].dropna().str.contains(r"  ", regex=True).any(), "Quedaron espacios dobles en DEPARTAMENTAL"

log.registrar(
    "DEPARTAMENTAL", f"Sin explorar a fondo en el diagnóstico ({antes_unicos} valores únicos)",
    "Trim + mayúsculas + colapso de espacios; vacíos a NA", 0,
    "Normalización defensiva estándar; no se detectaron variantes de escritura evidentes tras normalizar.",
)
print(f"DEPARTAMENTAL: {antes_unicos} -> {despues_unicos} categorías únicas tras normalizar.")


[Fase 1 - Identificadores] DEPARTAMENTAL: Trim + mayúsculas + colapso de espacios; vacíos a NA -> 0 registros afectados
DEPARTAMENTAL: 26 -> 26 categorías únicas tras normalizar.


## NIVEL, SECTOR, AREA, STATUS, MODALIDAD, JORNADA

Las seis son de dominio cerrado y ya llegaron limpias (ver diagnóstico). Se normalizan defensivamente y se **verifica** que el conjunto de categorías sea exactamente el esperado — si apareciera una categoría nueva, el `assert` lo detendría aquí mismo.


In [8]:
DOMINIOS_ESPERADOS = {
    "NIVEL": {"DIVERSIFICADO"},
    "SECTOR": {"OFICIAL", "PRIVADO", "MUNICIPAL", "COOPERATIVA"},
    "AREA": {"URBANA", "RURAL", "SIN ESPECIFICAR"},
    "STATUS": {"ABIERTA", "CERRADA DEFINITIVAMENTE", "CERRADA TEMPORALMENTE", "TEMPORAL NOMBRAMIENTO", "TEMPORAL TITULOS"},
    "MODALIDAD": {"MONOLINGUE", "BILINGUE"},
    "JORNADA": {"DOBLE", "INTERMEDIA", "MATUTINA", "NOCTURNA", "SIN JORNADA", "VESPERTINA"},
}

for col, dominio in DOMINIOS_ESPERADOS.items():
    df[col] = normalizar_espacios(df[col]).str.upper()
    valores_reales = set(df[col].unique())
    inesperados = valores_reales - dominio
    assert not inesperados, f"{col} tiene categorías fuera del dominio esperado: {inesperados}"
    df[col] = df[col].astype("category")

    log.registrar(
        col, "Ninguno (dominio cerrado, ya consistente)",
        "Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado para una variable categórica de dominio cerrado, en vez de texto genérico)", 0,
        f"Se verifica que el conjunto de categorías sea exactamente {sorted(dominio)}; SIN ESPECIFICAR/SIN JORNADA se mantienen como categoría propia, no se recodifican a NA (ver docs/02).",
    )

print("NIVEL/SECTOR/AREA/STATUS/MODALIDAD/JORNADA: OK, dominio verificado y casteado a category.")


[Fase 1 - Identificadores] NIVEL: Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado para una variable categórica de dominio cerrado, en vez de texto genérico) -> 0 registros afectados
[Fase 1 - Identificadores] SECTOR: Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado para una variable categórica de dominio cerrado, en vez de texto genérico) -> 0 registros afectados
[Fase 1 - Identificadores] AREA: Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado para una variable categórica de dominio cerrado, en vez de texto genérico) -> 0 registros afectados
[Fase 1 - Identificadores] STATUS: Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado para una variable categórica de dominio cerrado, en vez de texto genérico) -> 0 registros afectados
[Fase 1 - Identificadores] MODALIDAD: Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado para una variable categórica de dominio cerrado, en vez de t

## PLAN

13 categorías, 4 de ellas variantes de `SEMIPRESENCIAL` potencialmente redundantes. **No se unifican automáticamente** (decisión que requiere criterio de dominio, no solo de texto — ver riesgos en el plan); se deja el detalle original intacto.


In [9]:
PLAN_ESPERADO = {
    "A DISTANCIA", "DIARIO(REGULAR)", "DOMINICAL", "FIN DE SEMANA", "INTERCALADO", "IRREGULAR", "MIXTO",
    "SABATINO", "SEMIPRESENCIAL", "SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)", "SEMIPRESENCIAL (FIN DE SEMANA)",
    "SEMIPRESENCIAL (UN DÍA A LA SEMANA)", "VIRTUAL A DISTANCIA",
}

df["PLAN"] = normalizar_espacios(df["PLAN"]).str.upper()
valores_reales = set(df["PLAN"].unique())
assert valores_reales == PLAN_ESPERADO, f"PLAN cambió de dominio: {valores_reales.symmetric_difference(PLAN_ESPERADO)}"
df["PLAN"] = df["PLAN"].astype("category")

log.registrar(
    "PLAN", "13 categorías, 4 variantes de SEMIPRESENCIAL potencialmente redundantes",
    "Trim + mayúsculas defensivo; casteo a category. NO se unifican las variantes de SEMIPRESENCIAL: se preserva el detalle original", 0,
    "Unificar sin verificar con criterio de dominio (no solo de texto) arriesga perder información real sobre la modalidad (ver docs/02_Plan_de_Limpieza.md).",
)
print("PLAN: OK, dominio verificado (13 categorías), sin unificar.")


[Fase 1 - Identificadores] PLAN: Trim + mayúsculas defensivo; casteo a category. NO se unifican las variantes de SEMIPRESENCIAL: se preserva el detalle original -> 0 registros afectados
PLAN: OK, dominio verificado (13 categorías), sin unificar.


## ARCHIVO_ORIGEN (columna técnica, no es una variable del fenómeno)

In [10]:
assert df["ARCHIVO_ORIGEN"].map(es_vacio).sum() == 0, "ARCHIVO_ORIGEN no debería tener vacíos (se agregó nosotros mismos al descargar)"

log.registrar(
    "ARCHIVO_ORIGEN", "Ninguno",
    "Sin cambios; se conserva para trazabilidad/auditoría del proceso de descarga", 0,
    "Es un metadato técnico nuestro, no una variable del fenómeno; se documentará como tal en el Libro de Códigos.",
)
print("ARCHIVO_ORIGEN: OK, sin cambios.")


[Fase 1 - Identificadores] ARCHIVO_ORIGEN: Sin cambios; se conserva para trazabilidad/auditoría del proceso de descarga -> 0 registros afectados
ARCHIVO_ORIGEN: OK, sin cambios.


## Guardar salida de la Fase 1

In [11]:
os.makedirs(CARPETA_INTERIM, exist_ok=True)

ruta_salida = os.path.join(CARPETA_INTERIM, "fase1_identificadores.csv")
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

ruta_log = os.path.join(CARPETA_INTERIM, "log_fase1.csv")
log.guardar(ruta_log)

print(f"\nFase 1 completa: {df.shape[0]} filas, {df.shape[1]} columnas -> {ruta_salida}")
log.a_dataframe()


Registro de transformaciones de Fase 1 - Identificadores guardado en: C:\Users\jplop\Downloads\DATAS\scripts\..\data\interim\log_fase1.csv

Fase 1 completa: 11867 filas, 18 columnas -> C:\Users\jplop\Downloads\DATAS\scripts\..\data\interim\fase1_identificadores.csv


,fase,variable,problema_detectado,transformacion,registros_afectados,justificacion
0,Fase 1 - Identificadores,(todas),"23 filas 100% vacías (una por archivo/departamento, artefacto del GridView d...",Eliminar filas completamente vacías,23,No aportan ningún dato y distorsionan cualquier estadística; documentadas en...
1,Fase 1 - Identificadores,CODIGO,Ninguno (ya cumple el patrón y es único en el 100% de las filas),Sin cambios; se verifica el patrón ##-##-####-## y la unicidad como prueba d...,0,No hay nada que corregir; se documenta la verificación para la Actividad 7.
2,Fase 1 - Identificadores,DISTRITO,532 celdas vacías; 3 formatos distintos conviviendo en el campo,Trim/colapso de espacios y conversión de vacíos a NA explícito; se preserva ...,532,Forzar un solo formato numérico inventaría dígitos que no están en la fuente...
3,Fase 1 - Identificadores,DEPARTAMENTO,Ninguno (100% dentro del catálogo oficial del propio sitio),Trim + mayúsculas defensivo; se valida contra data/catalogo_departamentos_mu...,0,"Campo de <select> en la fuente, no texto libre digitado; se mantienen GUATEM..."
4,Fase 1 - Identificadores,MUNICIPIO,"Ninguno (100% de los pares (DEPARTAMENTO, MUNICIPIO) existen en el catálogo ...",Trim + mayúsculas defensivo; se valida el par completo contra el catálogo,0,"Mismo argumento que DEPARTAMENTO: campo de <select>, no texto libre."
5,Fase 1 - Identificadores,DEPARTAMENTAL,Sin explorar a fondo en el diagnóstico (26 valores únicos),Trim + mayúsculas + colapso de espacios; vacíos a NA,0,Normalización defensiva estándar; no se detectaron variantes de escritura ev...
6,Fase 1 - Identificadores,NIVEL,"Ninguno (dominio cerrado, ya consistente)",Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado par...,0,Se verifica que el conjunto de categorías sea exactamente ['DIVERSIFICADO'];...
7,Fase 1 - Identificadores,SECTOR,"Ninguno (dominio cerrado, ya consistente)",Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado par...,0,"Se verifica que el conjunto de categorías sea exactamente ['COOPERATIVA', 'M..."
8,Fase 1 - Identificadores,AREA,"Ninguno (dominio cerrado, ya consistente)",Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado par...,0,"Se verifica que el conjunto de categorías sea exactamente ['RURAL', 'SIN ESP..."
9,Fase 1 - Identificadores,STATUS,"Ninguno (dominio cerrado, ya consistente)",Trim + mayúsculas defensivo; se castea a tipo 'category' (tipo apropiado par...,0,"Se verifica que el conjunto de categorías sea exactamente ['ABIERTA', 'CERRA..."
